# Rescoring the Zero-Shot Baselines

The first scoring code (v1) mapped Devanagari loanwords to English inside `for_scoring`, so WER already accepted English words in either script and T-WER always equalled WER. This notebook scores every saved zero-shot output again with the fixed code (v2):

1. **WER** is strict. Both texts get the normal clean-up (NFC, lowercase, punctuation removed, numbers as digits), but no loanword mapping, so कस्टम for "custom" is an error.
2. **T-WER** follows MUCS 2021. An English word in the reference is correct if the output has it in Latin script or in an accepted Devanagari form from `data/lists/transliterations.csv`.

It reads only the saved `results/<run>/hyps.jsonl` files and runs no model. Runs that are not on disk yet are listed and skipped.

- **Inputs:** `data/lists/transliterations_review.csv`, `data/lists/loanwords.csv`, the test manifest and `results/*/hyps.jsonl`.
- **Outputs:** `data/lists/transliterations.csv`, and v2 rows in `results/summary.csv`. The v1 rows are kept; the v2 rows are replaced on every run.

## Imports and Paths

In [1]:
import csv
import datetime
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

PROJECT_ROOT = Path.cwd().parents[1]  # notebooks/analysis -> project root
LISTS = PROJECT_ROOT / "data" / "lists"
RESULTS = PROJECT_ROOT / "results"

## T-WER Mapping

The review file has one row per English word and Devanagari form, sorted by how often the word appears in the test references. The mapping keeps the rows with `accept` = 1. After a hand check of the review file, this notebook is run again to rebuild the mapping and the scores.

In [2]:
review = pd.read_csv(LISTS / "transliterations_review.csv", keep_default_na=False)
mapping = review[review["accept"] == 1][["english", "devanagari"]]
mapping.to_csv(LISTS / "transliterations.csv", index=False)
print(f"{len(mapping)} forms for {mapping['english'].nunique()} English words; {len(review) - len(mapping)} review rows not accepted")
print(review[review["accept"] == 0]["note"].value_counts().to_string())

2489 forms for 1845 English words; 119 review rows not accepted
note
no form: joined token, file name or code    92
normal Hindi word                           27


In [3]:
norm = Normalizer(LISTS / "loanwords.csv", LISTS / "transliterations.csv")

with open(PROJECT_ROOT / "data/processed/openslr104/test_manifest.jsonl", encoding="utf-8") as f:
    manifest = [json.loads(line) for line in f]
refs = [r["text"] for r in manifest]
assert score(refs, refs, norm) == {"wer": 0.0, "t_wer": 0.0}

## Runs

Each run folder, with the model and decoding label used in `results/summary.csv`. Nemotron has three language prompts at two latencies.

In [4]:
FALLBACK = "temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0)"
QWEN = "qwen-asr 0.0.6 defaults (greedy, max 512 new tokens, repetition clean-up), language=None, bf16, whole clips"
RUNS = {
    "whisper-small-zeroshot": ("openai/whisper-small", f"{FALLBACK}, language=hi, fp16, 30 s chunks"),
    "tara-zeroshot": ("Trelis/tara", f"{FALLBACK}, language=hi + mixedcode, bf16, 30 s pieces"),
    "qwen3-asr-0.6b-zeroshot": ("Qwen/Qwen3-ASR-0.6B", QWEN),
    "srota-zeroshot": ("Surajgameramp/srota", QWEN),
}
NEMOTRON = {
    (language, ms): f"nemotron-3.5-asr-streaming-0.6b-zeroshot{'' if language == 'auto' else '-' + language}-{ms}ms"
    for language in ("auto", "hi-IN", "en-US")
    for ms in (560, 1120)
}
for (language, ms), folder in NEMOTRON.items():
    RUNS[folder] = ("nvidia/nemotron-3.5-asr-streaming-0.6b", f"greedy RNN-T, streaming latency {ms} ms, language={language}, float32, whole clips")

hyps = {}
for folder in RUNS:
    path = RESULTS / folder / "hyps.jsonl"
    if not path.exists():
        print("not on disk yet (run on Colab):", folder)
        continue
    with open(path, encoding="utf-8") as f:
        rows = [json.loads(line) for line in f]
    assert [r["utt_id"] for r in rows] == [r["utt_id"] for r in manifest], folder
    assert [r["ref"] for r in rows] == refs, folder
    hyps[folder] = rows
print(len(hyps), "runs loaded")

not on disk yet (run on Colab): nemotron-3.5-asr-streaming-0.6b-zeroshot-hi-IN-560ms
not on disk yet (run on Colab): nemotron-3.5-asr-streaming-0.6b-zeroshot-hi-IN-1120ms
not on disk yet (run on Colab): nemotron-3.5-asr-streaming-0.6b-zeroshot-en-US-560ms
not on disk yet (run on Colab): nemotron-3.5-asr-streaming-0.6b-zeroshot-en-US-1120ms
6 runs loaded


## Raw Output Check

Scoring must start from the raw model output. A raw output keeps case, punctuation or extra spaces, so clean-up changes it, and it keeps Devanagari loanwords that `normalize` would have turned into English. A run with zero in both columns was saved after normalization.

In [5]:
check = pd.DataFrame([
    {
        "run": folder,
        "changed by clean-up": sum(norm.clean(r["hyp"]) != r["hyp"] for r in rows),
        "Devanagari loanwords": sum(w in norm.loanwords for r in rows for w in norm.clean(r["hyp"]).split()),
        "empty": sum(not r["hyp"].strip() for r in rows),
    }
    for folder, rows in hyps.items()
])
print(check.to_string(index=False))

                                            run  changed by clean-up  Devanagari loanwords  empty
                         whisper-small-zeroshot                 2939                   943      0
                                  tara-zeroshot                 2807                    74      0
                        qwen3-asr-0.6b-zeroshot                 1454                  3065      4
                                 srota-zeroshot                   97                  1135      0
 nemotron-3.5-asr-streaming-0.6b-zeroshot-560ms                 2678                  7234     14
nemotron-3.5-asr-streaming-0.6b-zeroshot-1120ms                 2496                  7357     20


## Scores

In [6]:
scores = {folder: score(refs, [r["hyp"] for r in rows], norm) for folder, rows in hyps.items()}
table = pd.DataFrame([{"run": f, "wer": round(s["wer"], 4), "t_wer": round(s["t_wer"], 4)} for f, s in scores.items()])
assert (table["t_wer"] <= table["wer"]).all()
print(table.to_string(index=False))

                                            run    wer  t_wer
                         whisper-small-zeroshot 0.9344 0.9078
                                  tara-zeroshot 0.2966 0.2947
                        qwen3-asr-0.6b-zeroshot 0.5022 0.4209
                                 srota-zeroshot 0.3280 0.3037
 nemotron-3.5-asr-streaming-0.6b-zeroshot-560ms 0.6048 0.4027
nemotron-3.5-asr-streaming-0.6b-zeroshot-1120ms 0.6040 0.3985


## Nemotron Language Prompt

Language tags the model wrote in each run (`no tag` when it wrote none), then WER and T-WER per prompt. The recommended prompt has the lowest T-WER at each latency.

In [7]:
for (language, ms), folder in NEMOTRON.items():
    if folder in hyps:
        print(f"{language} {ms} ms:", Counter(r["language"] or "no tag" for r in hyps[folder]).most_common())

auto 560 ms: [('hi-IN', 2603), ('no tag', 333), ('en-US', 6)]
auto 1120 ms: [('hi-IN', 2421), ('no tag', 517), ('en-US', 4)]


In [8]:
prompts = pd.DataFrame([
    {"prompt": language, "latency_ms": ms, "wer": round(scores[folder]["wer"], 4), "t_wer": round(scores[folder]["t_wer"], 4)}
    for (language, ms), folder in NEMOTRON.items()
    if folder in scores
])
print(prompts.to_string(index=False), "\n")
for ms, group in prompts.groupby("latency_ms"):
    best = group.loc[group["t_wer"].idxmin()]
    print(f"{ms} ms: lowest T-WER with {best['prompt']} ({best['t_wer']:.4f}), from {len(group)} of 3 prompts")

prompt  latency_ms    wer  t_wer
  auto         560 0.6048 0.4027
  auto        1120 0.6040 0.3985 

560 ms: lowest T-WER with auto (0.4027), from 1 of 3 prompts
1120 ms: lowest T-WER with auto (0.3985), from 1 of 3 prompts


## Results File

The v1 rows stay as they were. The v2 rows are written again from the scores above.

In [9]:
summary = RESULTS / "summary.csv"
with open(summary, encoding="utf-8") as f:
    kept = [r for r in csv.DictReader(f) if r["scoring"] != "v2"]

new = [
    {
        "model": RUNS[folder][0],
        "dataset": "openslr104",
        "split": "test",
        "n_utts": len(refs),
        "wer": round(s["wer"], 4),
        "t_wer": round(s["t_wer"], 4),
        "decoding": f"zero-shot, {RUNS[folder][1]}",
        "date": datetime.date.today().isoformat(),
        "scoring": "v2",
    }
    for folder, s in scores.items()
]
with open(summary, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(new[0]))
    writer.writeheader()
    writer.writerows(kept + new)
print(summary.read_text(encoding="utf-8"))

model,dataset,split,n_utts,wer,t_wer,decoding,date,scoring
openai/whisper-small,openslr104,test,2942,0.915,0.915,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi, fp16, 30 s chunks",2026-11-23,v1
Trelis/tara,openslr104,test,2942,0.295,0.295,"zero-shot, temperature fallback 0.0-1.0 (compression ratio 1.35, logprob -1.0, seed 0), language=hi + mixedcode, bf16, 30 s pieces",2026-11-23,v1
Qwen/Qwen3-ASR-0.6B,openslr104,test,2942,0.4362,0.4362,"zero-shot, qwen-asr 0.0.6 defaults (greedy, max 512 new tokens, repetition clean-up), language=None, bf16, whole clips",2026-11-23,v1
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4584,0.4584,"zero-shot, greedy RNN-T, streaming latency 560 ms, language=auto, float32, whole clips",2026-11-23,v1
nvidia/nemotron-3.5-asr-streaming-0.6b,openslr104,test,2942,0.4546,0.4546,"zero-shot, greedy RNN-T, streaming latency 1120 ms, language=auto, float32, whole clips",2026-11-23,v1
Surajgameramp/s